<h1 align="center">Analise</h1>

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import pandas as pd

%matplotlib inline

In [ ]:
def resolve_support_images() -> Path:
    """Find support_images next to cwd or one level up (e.g. running from notebook/)."""
    cwd = Path.cwd().resolve()
    for base in (cwd, cwd.parent):
        candidate = base / "support_images"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find support_images/. Run the notebook from the project root or from notebook/."
    )

In [ ]:
SUPPORT_IMAGES = resolve_support_images()
print(SUPPORT_IMAGES)

In [ ]:
def find_csv_files(root: Path) -> list[Path]:
    return sorted(root.rglob("*.csv"))

In [ ]:
raw = find_csv_files(SUPPORT_IMAGES)

valid_data = [
    "20260504_005306_preds_imagem_regular_200_epocas_10_folds_dice",
    "20260510_191013_preds_vessel_regular_200_epocas_10_folds_dice",
    "20260504_081718_preds_imagem_regular_200_epocas_10_folds_bce",
    "20260511_165927_preds_vessel_regular_200_epocas_10_folds_bce",
    "20260510_024020_preds_imagem_regular_200_epocas_10_folds_dice_bce_swap_0",
    "20260515_001931_preds_vessel_regular_200_epocas_10_folds_dice_bce_swap_0",
    "20260504_231233_preds_imagem_edge_200_epocas_10_folds_dice",
    "20260513_035403_preds_vessel_edge_200_epocas_10_folds_dice",
    "20260517_045850_preds_imagem_edge_200_epocas_10_folds_dice_unet_simples",
    "20260515_205409_preds_vessel_regular_200_epocas_10_folds_dice_unet_simples",
    "20260505_081932_preds_imagem_edge_200_epocas_10_folds_dice_unet_simples_informacao_de_borda",
    "20260515_103607_preds_vessel_edge_200_epocas_10_folds_dice_unet_simples_informacao_de_borda",
    "20260505_123138_preds_imagem_regular_200_epocas_10_folds_dice_unet_simples_menos_camadas",
    "20260513_190824_preds_vessel_regular_200_epocas_10_folds_dice_unet_simples_menos_camadas",
    "20260516_141125_preds_imagem_edge_200_epocas_10_folds_dice_unet_simples_menos_camadas",
    "20260516_192822_preds_vessel_edge_200_epocas_10_folds_dice_unet_simples_menos_camadas"
]

csv_paths = list(filter(lambda x: x.parents[0].stem in valid_data, raw))
print(f"Found {len(csv_paths)} CSV file(s)")
for p in csv_paths:
    print(" ", p.relative_to(SUPPORT_IMAGES))

In [ ]:
def load_csv(path: Path, **read_csv_kwargs) -> pd.DataFrame:
    return pd.read_csv(path, **read_csv_kwargs)

In [ ]:
def load_all_csv(root: Path, **read_csv_kwargs) -> dict[str, pd.DataFrame]:
    out: dict[str, pd.DataFrame] = {}
    for path in find_csv_files(root):
        if path.parents[0].stem in valid_data:
            key = str(path.relative_to(root))
            out[key] = load_csv(path, **read_csv_kwargs)
    return out

In [ ]:
tables = load_all_csv(SUPPORT_IMAGES)
list(tables.keys())

In [ ]:
FIGSIZE = (10, 4)
EPOCH_COL = "epoch"
# Typical names from this project's logs: dice_test / loss_test (or dice_train / loss_train)
DICE_COL = "dice_test"
LOSS_COL = "loss_test"
DICE_COL_TRAIN = "dice_train"
LOSS_COL_TRAIN = "loss_train"
AGGREGATE_BY_EPOCH = True  # if True, mean Dice/Loss per epoch when multiple rows share an epoch

def _mark_last_point(ax, epochs, series, line) -> tuple[float, str] | None:
    if len(series) == 0 or pd.isna(series.iloc[-1]):
        return None
    xv, yv = epochs.iloc[-1], series.iloc[-1]
    color = line.get_color()
    ax.scatter([xv], [yv], s=50, zorder=5, color=color, edgecolors="white", linewidths=0.5)
    return yv, color


def _legends(ax, last_entries: list[tuple[str, float, str]]) -> None:
    leg_series = ax.legend(loc="best")
    if not last_entries:
        return
    handles = [
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor=color,
            markeredgecolor=color,
            markersize=8,
            label=f"{name}: {value:.4f}",
        )
        for name, value, color in last_entries
    ]
    ax.legend(handles=handles, title="Última época", loc="center right", framealpha=0.9)
    ax.add_artist(leg_series)


def plot_epoch_metrics(rel_path: str, df: pd.DataFrame) -> None:
    cols_needed = [EPOCH_COL, DICE_COL, LOSS_COL, DICE_COL_TRAIN, LOSS_COL_TRAIN]
    missing = [c for c in cols_needed if c not in df.columns]
    if missing:
        print(f"{rel_path}: missing columns {missing}. Available: {list(df.columns)}\n")
        return

    work = df[cols_needed].copy()
    for c in (DICE_COL, LOSS_COL, DICE_COL_TRAIN, LOSS_COL_TRAIN):
        work[c] = pd.to_numeric(work[c], errors="coerce")

    if AGGREGATE_BY_EPOCH:
        plot_df = work.groupby(EPOCH_COL, as_index=False)[[DICE_COL, LOSS_COL, DICE_COL_TRAIN, LOSS_COL_TRAIN]].mean().sort_values(EPOCH_COL)
    else:
        plot_df = work.sort_values(EPOCH_COL)

    file_path = rel_path.split("/")[0]
    epochs = plot_df[EPOCH_COL]

    fig, ax = plt.subplots(figsize=FIGSIZE)
    line_train, = ax.plot(epochs, plot_df[DICE_COL_TRAIN], marker="o", markersize=3, linewidth=1, label="treino")
    line_test, = ax.plot(epochs, plot_df[DICE_COL], marker="o", markersize=3, linewidth=1, label="teste")
    last_entries = []
    marked = _mark_last_point(ax, epochs, plot_df[DICE_COL_TRAIN], line_train)
    if marked is not None:
        last_entries.append(("treino", *marked))
    marked = _mark_last_point(ax, epochs, plot_df[DICE_COL], line_test)
    if marked is not None:
        last_entries.append(("teste", *marked))
    ax.set_title(f"{rel_path} — Dice vs epoch")
    # ax.set_title(f"Modelo otimizado por BCE — DSC vs Época")
    ax.set_xlabel("Épocas")
    ax.set_ylabel("DSC")
    _legends(ax, last_entries)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    savefig_1 = f"../support_images/{file_path}‌/dsc_vs_epoch.png".encode('ascii', 'ignore')
    plt.savefig(savefig_1)
    plt.show()

    fig, ax = plt.subplots(figsize=FIGSIZE)
    line_train, = ax.plot(epochs, plot_df[LOSS_COL_TRAIN], marker="o", markersize=3, linewidth=1, label="treino")
    line_test, = ax.plot(epochs, plot_df[LOSS_COL], marker="o", markersize=3, linewidth=1, color="C1", label="teste")
    last_entries = []
    marked = _mark_last_point(ax, epochs, plot_df[LOSS_COL_TRAIN], line_train)
    if marked is not None:
        last_entries.append(("treino", *marked))
    marked = _mark_last_point(ax, epochs, plot_df[LOSS_COL], line_test)
    if marked is not None:
        last_entries.append(("teste", *marked))
    ax.set_title(f"{rel_path} — Perda vs Época")
    # ax.set_title(f"Modelo otimizado por BCE — Perda vs Época")
    ax.set_xlabel("Épocas")
    ax.set_ylabel("Perdas")
    _legends(ax, last_entries)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    savefig_2 = f"../support_images/{file_path}‌/loss_vs_epoch.png".encode('ascii', 'ignore')
    plt.savefig(savefig_2)
    plt.show()


if not tables:
    print("No CSV files under support_images yet. Add some, then re-run the discovery cells.")
else:
    for rel_path, df in tables.items():
        print(f"\n--- {rel_path} ---")
        display(df.head())
        plot_epoch_metrics(rel_path, df)